# Diningcode Rating Extractor (Playwright Version)

## Installation
Run the following once in your terminal before opening Jupyter:
```
pip install playwright beautifulsoup4 pandas
playwright install chromium
```

In [ ]:
import re
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
import pandas as pd

In [ ]:
# Initialize data storage lists
item_name = []
item_area = []
item_avg_rating = []
item_rating_num = []
item_spec_area = []

user_name = []
user_tot_avg_rating = []
user_tot_rating_num = []
user_tot_follow_num = []

user_rating = []
user_query = []
taste = []
price = []
service = []
menu = []
date = []

In [ ]:
def get_item_contents(soup_detail):
    """Extract restaurant-level info from the detail page soup."""
    item_name.append(soup_detail.select('h1')[0].text)
    item_area.append(soup_detail.select('a.area')[0].text)
    item_avg_rating.append(soup_detail.select('span.point > strong')[0].text)
    item_rating_num.append(re.findall(r'\d+', soup_detail.select('span.point > span')[0].text))
    item_spec_area.append(soup_detail.select('span.profile_jibun')[0].text)


def full_access(detail_page, soup_detail):
    """
    Click 'Load more reviews' until all reviews are loaded,
    then extract each review's data.
    """
    c_rate = (int(''.join(re.findall(r'\d+', soup_detail.select('span.point > span')[0].text))) // 5)

    for i in range(c_rate):
        try:
            button = detail_page.locator('//*[@id="div_more_review"]')
            button.click(timeout=3000)
            detail_page.wait_for_timeout(500)
        except Exception:
            print('----delay or overlay----')
            try:
                detail_page.mouse.click(200, 200)
            except Exception:
                pass
            detail_page.wait_for_timeout(3000)
            continue

    html_all_reviews = detail_page.content()
    soup_all_reviews = BeautifulSoup(html_all_reviews, 'html.parser')

    for review in soup_all_reviews.select('div.latter-graph'):
        get_item_contents(soup_detail)

        user_name.append(review.select('p > span')[0].text)
        user_tot_avg_rating.append(review.select('p > span.info > span')[0].text)
        user_tot_rating_num.append(review.select('p > span.info > span')[1].text)
        user_tot_follow_num.append(review.select('p > span.info > span')[2].text)

        try:
            user_rating.append(review.select('span.total_score')[0].text)
        except Exception:
            user_rating.append("nan")

        try:
            user_query.append(review.select('div.review_contents.btxt')[0].text)
        except Exception:
            user_query.append("nan")

        try:
            taste.append(review.select('span.sub_title')[0].text)
        except Exception:
            taste.append("nan")

        try:
            price.append(review.select('span.sub_title')[1].text)
        except Exception:
            price.append('nan')

        try:
            service.append(review.select('span.sub_title')[2].text)
        except Exception:
            service.append('nan')

        try:
            menu.append(review.select('p.ordered_menu_list')[0].text)
        except Exception:
            menu.append("nan")

        try:
            date.append(review.select('span.date')[0].text)
        except Exception:
            date.append("nan")

In [ ]:
# sync_playwright() runs entirely synchronously — no asyncio, no event loop issues.
# This is the correct approach for Playwright inside Jupyter on Windows.

with sync_playwright() as p:
    browser = p.chromium.launch(
        headless=False,  # Set True to run in background
        args=[
            '--disable-dev-shm-usage',
            '--disable-notifications',
            '--disable-popup-blocking'
        ]
    )

    context = browser.new_context()
    page = context.new_page()

    url = 'https://www.diningcode.com/foodrank/%EC%84%9C%EC%9A%B8'
    page.goto(url)
    page.wait_for_timeout(5000)

    # ── Scroll down to load all category sections ──
    for _ in range(15):
        page.keyboard.press('PageDown')
        page.wait_for_timeout(1000)

    html = page.content()
    soup = BeautifulSoup(html, 'html.parser')

    for item in soup.select('h2.Card__Section__Title'):
        if "33" in item.text:
            break
        print(item.text)

    page.wait_for_timeout(2000)
    html = page.content()
    soup = BeautifulSoup(html, 'html.parser')

    # ── Main crawling loop ──
    for i in range(1, 33):
        try:
            k = len(soup.select(
                f'#root > main > div > section:nth-child({i}) > div:nth-of-type(2) > div:nth-of-type(2) > ul'
            )[0])
        except Exception:
            k = len(soup.select(
                '#root > main > div > section:nth-child(1) '
                '> div.sc-jaXxmE.bbcCvK.Card__Section__Wrap '
                '> div.sc-kbhJrz.hwQOdA.Card__Slide__Wrap > ul'
            )[0])

        print(k)

        for j in range(1, k):
            try:
                # Locate the restaurant link
                try:
                    detail_link = page.locator(
                        f'xpath=/html/body/div/main/div/section[{i}]/div[2]/div[2]/ul/li[{j}]/a'
                    )
                except Exception:
                    detail_link = page.locator(
                        f'xpath=/html/body/div/main/div/section[{i}]/div[3]/div[2]/ul/li[{j}]/a'
                    )

                # Capture the new tab that opens when pressing Enter
                with context.expect_page() as new_page_info:
                    detail_link.press('Enter')
                    page.wait_for_timeout(3000)

                detail_page = new_page_info.value
                detail_page.wait_for_load_state('domcontentloaded')

                print('post url : ', page.url)
                print('changed url : ', detail_page.url)

                html_detail = detail_page.content()
                soup_detail = BeautifulSoup(html_detail, 'html.parser')

                full_access(detail_page, soup_detail)

                # Close detail tab; focus returns to listing page automatically
                detail_page.close()

            except Exception as e:
                print(f'Error at section {i}, item {j}: {e}')
                for stray in context.pages[1:]:
                    stray.close()
                page.wait_for_timeout(500)

    browser.close()

In [ ]:
data = [
    item_name, item_area, item_avg_rating, item_spec_area,
    user_name, user_tot_avg_rating, user_tot_rating_num, user_tot_follow_num,
    user_rating, user_query, taste, price, service, menu, date
]

col = [
    'item_name', 'item_area', 'item_avg_rating', 'item_spec_area',
    'user_name', 'user_tot_avg_rating', 'user_tot_rating_num', 'user_tot_follow_num',
    'user_rating', 'user_query', 'taste', 'price', 'service', 'menu', 'date'
]

df = pd.DataFrame(data, index=col).transpose()

In [ ]:
print(df.shape)
df.sample(19)

In [ ]:
df.to_csv('diningcode_data_crawling_playwright.csv', index=False)